<a href="https://colab.research.google.com/github/thanhsonnguyen-gif/ai-text-detection/blob/main/notebooks/E2b_normalize_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2b · Chuẩn hóa v2 · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.2, RQ1, phân tích bổ sung)
**Vấn đề:** ở E2, bản defended (chuẩn hóa v1) trên tấn công homoglyph của RAID còn FPR 0,082 và AUROC 0,907, kém văn bản sạch (0,063 / 0,913).

**Nguyên nhân (đọc từ mã nguồn RAID):** bộ sinh homoglyph của RAID (`liamdugan/raid`, `generation/adversarial/attackers/homoglyph.py`)
thay chữ Latin bằng Cyrillic **hoặc chữ HOA Hy Lạp** (Α Β Ε Κ Ο Ρ Μ Η Τ Χ Ι Ν Ζ) và Cyrillic `І`. Chuẩn hóa v1 chỉ biết 19 ký tự Cyrillic;
`lower()` biến `Α` thành `α` (Hy Lạp), không thành `a`, nên các từ viết hoa vẫn bị vỡ.

**v2** = bảng v1 + 14 ký tự đó (`textops.EXTRA_FROM`). v1 vẫn là mặc định để kết quả E1–E4 tái lập được.

**Kỷ luật thí nghiệm:** bảng v2 lấy từ **đặc tả của bộ tấn công**, không chọn theo số liệu test; mô hình v2 huấn luyện trên train,
ngưỡng chọn trên val, test chỉ dùng để báo cáo. Vì động cơ sửa đến từ một kết quả trên test, báo cáo v2 là **phân tích bổ sung**,
kết quả chính của RQ1 vẫn là v1.

Chạy: **Runtime → Run all** (~25 phút).

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:   # --ff-only + check=True: pull hỏng thì DỪNG, không âm thầm chạy tiếp với code cũ
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q", "--ff-only"], check=True)
print("Code đang dùng:", subprocess.run(["git", "-C", "/content/repo", "log", "-1", "--format=%h %s (%cd)", "--date=short"],
                                        capture_output=True, text=True).stdout.strip())
for k in [k for k in sys.modules if k == "aitext" or k.startswith("aitext.")]:
    del sys.modules[k]   # chạy lại BOOTSTRAP ⇒ nạp lại aitext mới nhất, không dùng bản cũ còn trong bộ nhớ
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

Mounted at /content/drive
Code đang dùng: 333f358 E2b: chuẩn hóa v2 (chữ HOA Hy Lạp + Cyrillic І theo bộ tấn công RAID) (2026-11-05)
Java 21 | PySpark 4.0.5
Drive: /content/drive/MyDrive/BigData_IE212_R11_VB2/AI-Text-Project


In [ ]:
from aitext import data, models, evaluate, textops
import unicodedata
train, val, test = data.load_splits(spark)
th = models.load_thresholds(); print(th)
m_v1 = models.load("lr_def")

{'reg_param': 1.0, 'thr_base': 0.7318487066775035, 'thr_def': 0.7309143970540265, 'fpr_target': 0.05}


In [ ]:
# 1. Chẩn đoán: ký tự ngoài ASCII còn sót sau chuẩn hóa v1 và v2 trong dữ liệu homoglyph (chỉ đếm ký tự, không dùng nhãn)
hg = data.load_attacked(spark, "homoglyph").cache()
def leftover(version, k=20):
    t = (hg.select(F.explode(F.split(F.regexp_replace(textops.normalize(F.col("generation"), version), r"[\x00-\x7F]", ""), "")).alias("c"))
         .filter("c != ''").groupBy("c").count().orderBy(F.desc("count")).limit(k).toPandas())
    t["tên Unicode"] = t["c"].map(lambda ch: unicodedata.name(ch, "?")); t.insert(0, "chuẩn hóa", f"v{version}")
    return t
t57a = pd.concat([leftover(1), leftover(2)], ignore_index=True); display(t57a)
evaluate.save_table(t57a, "T5.7a_homoglyph_leftover_chars",
                    notes="Top ký tự ngoài ASCII còn lại sau chuẩn hóa trong tập homoglyph (phía test). v2 phải hết chữ Hy Lạp/Cyrillic.")

# Tác dụng phụ trên dữ liệu sạch: bao nhiêu văn bản train chứa ký tự mà v2 sẽ đổi thêm
n_aff = train.filter(F.col("generation").rlike("[" + textops.EXTRA_FROM + "]")).count()
print(f"Văn bản train bị v2 đổi thêm: {n_aff} / {train.count()}")

,chuẩn hóa,c,count,tên Unicode
0,v1,ι,7819,GREEK SMALL LETTER IOTA
1,v1,і,7738,CYRILLIC SMALL LETTER BYELORUSSIAN-UKRAINIAN I
2,v1,τ,7595,GREEK SMALL LETTER TAU
3,v1,α,5915,GREEK SMALL LETTER ALPHA
4,v1,ν,3489,GREEK SMALL LETTER NU
5,v1,μ,3377,GREEK SMALL LETTER MU
6,v1,β,3366,GREEK SMALL LETTER BETA
7,v1,η,3157,GREEK SMALL LETTER ETA
8,v1,ρ,2655,GREEK SMALL LETTER RHO
9,v1,ε,1991,GREEK SMALL LETTER EPSILON


Văn bản train bị v2 đổi thêm: 21 / 58987


In [ ]:
# 2. Huấn luyện defended v2 trên TOÀN BỘ train (giống lr_def của E1), chọn ngưỡng trên VAL
tr = data.with_class_weight(train).cache()
t0 = time.time(); m_v2 = models.fit(tr, defended=True, norm_version=2); print(f"train {time.time() - t0:.0f}s")
models.save(m_v2, "lr_def_v2")
thr_v2 = evaluate.thr_at_fpr(evaluate.score(m_v2, val, models.text_col(True, 2)))
print(models.save_thresholds(thr_def_v2=thr_v2))      # thêm khóa mới, không đổi thr_def của v1

train 394s
{'reg_param': 1.0, 'thr_base': 0.7318487066775035, 'thr_def': 0.7309143970540265, 'fpr_target': 0.05, 'thr_def_v2': 0.7308515095522097}


In [ ]:
# 3. Đo trên TEST: sạch + 11 kiểu tấn công RAID, v1 và v2 cạnh nhau → T5.7
MV = {"v1": (m_v1, 1, th["thr_def"]), "v2": (m_v2, 2, thr_v2)}
rows = []
for atk in ["none"] + config.RAID_ATTACKS:
    d = (test if atk == "none" else data.load_attacked(spark, atk)).cache()
    row = {"attack": atk, "n": d.count()}
    for v, (m, ver, thr) in MV.items():
        p = evaluate.score(m, d, models.text_col(True, ver)).cache()
        row[f"AUROC {v}"] = evaluate.auroc(p)
        row[f"TPR {v}"] = evaluate.rate_above(p, 1, thr); row[f"FPR {v}"] = evaluate.rate_above(p, 0, thr)
        evaluate.log_run("E2b", OWNER, f"defended_{v}|{atk}",
                         {"auroc": row[f"AUROC {v}"], "tpr": row[f"TPR {v}"], "fpr": row[f"FPR {v}"], "threshold": thr, "n": row["n"]})
        p.unpersist()
    rows.append(row); d.unpersist()
t57 = pd.DataFrame(rows)
for k in ["AUROC", "TPR", "FPR"]:
    t57[f"Δ{k}"] = t57[f"{k} v2"] - t57[f"{k} v1"]
display(t57.round(3))
evaluate.save_table(t57, "T5.7_normalize_v2",
                    notes="Phân tích bổ sung. v2 = v1 + chữ HOA Hy Lạp và Cyrillic І theo mã nguồn bộ tấn công RAID. "
                          "Mô hình defended huấn luyện trên toàn bộ train; ngưỡng chọn trên val.")

,attack,n,AUROC v1,TPR v1,FPR v1,AUROC v2,TPR v2,FPR v2,ΔAUROC,ΔTPR,ΔFPR
0,none,15840,0.913,0.734,0.063,0.913,0.734,0.064,-0.000,0.000,0.001
1,homoglyph,3285,0.907,0.736,0.082,0.924,0.754,0.060,0.017,0.018,-0.022
2,number,3410,0.910,0.734,0.073,0.910,0.734,0.074,-0.000,0.000,0.001
3,article_deletion,3255,0.909,0.728,0.063,0.909,0.728,0.064,-0.000,0.000,0.001
4,insert_paragraphs,3344,0.907,0.731,0.073,0.907,0.731,0.074,0.000,0.000,0.001
5,perplexity_misspelling,3337,0.919,0.741,0.064,0.919,0.741,0.064,0.000,-0.001,0.000
6,upper_lower,3341,0.908,0.728,0.057,0.908,0.728,0.058,-0.000,0.000,0.001
7,whitespace,3375,0.915,0.746,0.059,0.915,0.746,0.061,-0.000,0.001,0.001
8,zero_width_space,3299,0.920,0.737,0.061,0.920,0.736,0.062,0.000,-0.001,0.001
9,synonym,3414,0.884,0.653,0.070,0.884,0.653,0.070,-0.000,0.000,0.000


## Ghi chú cho Chương 5
- ...